# SignBridge PSL — Word-Sign GRU Model Training

This notebook downloads the Dynamic Word-Level Pakistan Sign Language dataset
(pre-extracted MediaPipe landmarks), trains a GRU temporal model, and exports
it to TensorFlow.js for the browser.

**Expected runtime:** ~1–2 hours on Colab GPU (T4)
**Output:** `psl-gru-word-signs/` folder saved to Google Drive

---
### What this notebook does
1. Installs dependencies & clones the SignBridge repo
2. Downloads the Dynamic Word-Level PSL dataset from Kaggle
   (mohib123456/dynamic-word-level-pakistan-sign-language-dataset)
3. Parses pre-extracted MediaPipe landmarks into 30-frame windows
4. Trains a 2-layer GRU classifier (~300K params, same architecture as ASL)
5. Exports to TF.js LayersModel format
6. Saves everything to your Google Drive

### Prerequisites
- Kaggle API key (kaggle.json) uploaded to Colab
- GPU runtime: **Runtime → Change runtime type → T4 GPU**
- Google Drive with ~2 GB free space

In [ ]:
# ── Cell 1: Install dependencies ───────────────────────────────────────
!pip install -q tensorflow>=2.16.0 tensorflowjs>=4.22.0 scikit-learn
!pip install -q kaggle tqdm

import os, json, sys, shutil, glob
from collections import Counter
from pathlib import Path

print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/signbridge-psl-words'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Artifacts will be saved to: {DRIVE_OUTPUT}")

In [ ]:
# ── Cell 3: Clone SignBridge repo ──────────────────────────────────────
REPO_URL = 'https://github.com/mhmdtaha091/SignBridge.git'
REPO_DIR = '/content/SignBridge'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}/ml
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Cell 4: Upload Kaggle API key ──────────────────────────────────────
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/ 2>/dev/null || echo "⚠ kaggle.json not found. Upload it."
!chmod 600 ~/.kaggle/kaggle.json 2>/dev/null

try:
    !kaggle datasets list --sort-by=updated 2>/dev/null | head -3
    print("✅ Kaggle API key works!")
except:
    print("⚠ Kaggle API key not set up. Upload kaggle.json to /content/ and re-run.")

In [ ]:
# ── Cell 5: Download Dynamic Word-Level PSL dataset ────────────────────
DATASET = 'mohib123456/dynamic-word-level-pakistan-sign-language-dataset'
DATA_DIR = '/content/psl_dynamic_words'

if not os.path.exists(DATA_DIR):
    print(f"Downloading {DATASET}...")
    !kaggle datasets download -d {DATASET} -p /content/psl_words_zip --unzip
    # Find the extracted directory
    extracted = glob.glob('/content/psl_words_zip/*')
    if extracted:
        # Move to our expected path
        if os.path.isdir('/content/psl_words_zip'):
            shutil.move('/content/psl_words_zip', DATA_DIR)
        print(f"✅ Dataset extracted to {DATA_DIR}")
    else:
        print("Checking extraction structure...")
        !ls /content/psl_words_zip/ 2>/dev/null || echo "Nothing found"
else:
    print(f"✅ Dataset already at {DATA_DIR}")

# List what's in the dataset
print("\nDataset contents:")
!ls -la {DATA_DIR}/ 2>/dev/null | head -20

In [ ]:
# ── Cell 6: Parse the dataset format ────────────────────────────────────
# The Dynamic Word-Level PSL dataset has MediaPipe landmarks pre-extracted.
# Common format: .npy files with shape (num_frames, 126) or (num_frames, 63)
# or JSON files with {label: ..., landmarks: [[x,y,z],...]}
#
# Let's explore and parse.

import numpy as np

# Look for data files
npy_files = glob.glob(os.path.join(DATA_DIR, '**', '*.npy'), recursive=True)
json_files = glob.glob(os.path.join(DATA_DIR, '**', '*.json'), recursive=True)
csv_files = glob.glob(os.path.join(DATA_DIR, '**', '*.csv'), recursive=True)
npz_files = glob.glob(os.path.join(DATA_DIR, '**', '*.npz'), recursive=True)

print(f"Found: {len(npy_files)} .npy, {len(json_files)} .json, {len(csv_files)} .csv, {len(npz_files)} .npz files")

# Try to determine the structure
if npy_files:
    sample = np.load(npy_files[0])
    print(f"\nSample .npy file: {npy_files[0]}")
    print(f"  Shape: {sample.shape}")
    print(f"  Dtype: {sample.dtype}")

if json_files:
    with open(json_files[0]) as f:
        sample = json.load(f)
    print(f"\nSample .json file: {json_files[0]}")
    print(f"  Type: {type(sample)}")
    if isinstance(sample, dict):
        print(f"  Keys: {list(sample.keys())[:10]}")
    elif isinstance(sample, list):
        print(f"  Length: {len(sample)}")
        print(f"  First element: {str(sample[0])[:200]}")

if csv_files:
    import pandas as pd
    df = pd.read_csv(csv_files[0], nrows=5)
    print(f"\nSample .csv file: {csv_files[0]}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Shape: {df.shape}")
    print(f"\n{df.head()}")

In [ ]:
# ── Cell 7: Build training data from the PSL dataset ───────────────────
# This cell adapts to whatever format the dataset uses.
# We produce: X_train (N, 30, 159) and y_train (N,) arrays.

WINDOW_SIZE = 30  # Match the browser's sequence classifier
STRIDE = 10
FEATURE_SIZE = 159  # Full: both hands (63+63) + pose (33)

def parse_psl_dataset():
    """
    Parse the Dynamic Word-Level PSL dataset into training arrays.
    Handles multiple possible formats:
      - .npy: (frames, landmarks) per file, with folder = label
      - .json: [{landmarks, label}] 
      - .csv: rows of landmark data with label column
    """
    X = []
    y = []
    label_counts = Counter()

    # Check for .npy files organized in label folders
    label_dirs = [
        d for d in os.listdir(DATA_DIR)
        if os.path.isdir(os.path.join(DATA_DIR, d))
        and not d.startswith('.')
    ]

    if label_dirs:
        print(f"Found {len(label_dirs)} label directories")
        for label_dir in label_dirs[:100]:  # Cap at 100 labels
            label_path = os.path.join(DATA_DIR, label_dir)
            npy_files = glob.glob(os.path.join(label_path, '*.npy'))

            for npy_file in npy_files:
                try:
                    data = np.load(npy_file)
                    if data.ndim != 2 or data.shape[1] < 63:
                        continue

                    # Pad/crop to 159 features
                    if data.shape[1] < FEATURE_SIZE:
                        padded = np.zeros((data.shape[0], FEATURE_SIZE), dtype=np.float32)
                        padded[:, :data.shape[1]] = data
                        data = padded
                    elif data.shape[1] > FEATURE_SIZE:
                        data = data[:, :FEATURE_SIZE]

                    # Segment into windows
                    for start in range(0, max(1, data.shape[0] - WINDOW_SIZE + 1), STRIDE):
                        end = start + WINDOW_SIZE
                        window = data[start:end]
                        if window.shape[0] < WINDOW_SIZE:
                            # Pad with edge replication
                            pad_len = WINDOW_SIZE - window.shape[0]
                            window = np.concatenate([
                                window,
                                np.tile(window[-1:], (pad_len, 1))
                            ])
                        X.append(window.astype(np.float32))
                        y.append(label_dir)
                        label_counts[label_dir] += 1
                except Exception as e:
                    continue

    # Check for a single .npz with X/y arrays
    npz_files = glob.glob(os.path.join(DATA_DIR, '**', '*.npz'), recursive=True)
    if npz_files and not X:
        print(f"Trying .npz format...")
        for npz_file in npz_files:
            try:
                data = np.load(npz_file, allow_pickle=True)
                print(f"  Keys in {os.path.basename(npz_file)}: {list(data.keys())}")
                # Common keys: X, y, labels, landmarks, features
                for key in ['X', 'features', 'landmarks', 'data']:
                    if key in data:
                        X_arr = data[key]
                        print(f"  '{key}' shape: {X_arr.shape}")
                        break
            except:
                continue

    # Check for JSON files
    json_files = glob.glob(os.path.join(DATA_DIR, '**', '*.json'), recursive=True)
    if json_files and not X:
        for jf in json_files[:5]:
            with open(jf) as f:
                sample = json.load(f)
            print(f"  {os.path.basename(jf)}: {type(sample).__name__}")
            if isinstance(sample, dict):
                print(f"    keys: {list(sample.keys())[:10]}")

    return np.array(X, dtype=np.float32), np.array(y), label_counts

X_raw, y_raw, label_counts = parse_psl_dataset()
print(f"\nParsed dataset: {len(X_raw)} windows, {len(set(y_raw))} unique labels")
print(f"Feature shape: {X_raw.shape[1:] if len(X_raw) > 0 else 'N/A'}")

if label_counts:
    print("\nPer-label window counts (top 30):")
    for label, count in label_counts.most_common(30):
        print(f"  {label:30s} {count:5d}")

In [ ]:
# ── Cell 8: Verify GPU ──────────────────────────────────────────────────
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU available: {gpus}")
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else:
    print("⚠ No GPU detected. Training will work but be slower.")
    print("  Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── Cell 9: Train the PSL GRU model ────────────────────────────────────
# Architecture: GRU(128) → Dropout(0.3) → GRU(64) → Dropout(0.3) → Softmax
# Same architecture as the ASL word-sign model (~300K params).

if len(X_raw) == 0:
    print("❌ No training data found. Check Cell 7 output.")
    print("The dataset format may differ. Inspect files and update the parse function.")
else:
    # Encode labels
    unique_labels = sorted(set(y_raw))
    label_to_idx = {l: i for i, l in enumerate(unique_labels)}
    y_encoded = np.array([label_to_idx[l] for l in y_raw], dtype=np.int32)

    print(f"Classes: {len(unique_labels)}")
    print(f"Feature shape: {X_raw.shape}")
    print(f"Labels: {unique_labels}")

    # Train/val split
    from sklearn.model_selection import train_test_split
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_raw, y_encoded, test_size=0.15, random_state=42,
        stratify=y_encoded if len(set(y_encoded)) > 1 else None
    )

    # Build model (same architecture as ml/model.py)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(WINDOW_SIZE, FEATURE_SIZE), name='landmarks'),
        tf.keras.layers.GRU(128, return_sequences=True, name='gru1'),
        tf.keras.layers.Dropout(0.3, name='drop1'),
        tf.keras.layers.GRU(64, name='gru2'),
        tf.keras.layers.Dropout(0.3, name='drop2'),
        tf.keras.layers.Dense(len(unique_labels), activation='softmax', name='output'),
    ], name='signbridge_gru')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    model.summary()

    # Callbacks
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=15, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5
        ),
    ]

    # Train
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=80,
        batch_size=32,
        callbacks=callbacks,
        verbose=1,
    )

    # Best validation accuracy
    best_val_acc = max(history.history['val_accuracy'])
    print(f"\n✅ Training complete — best val accuracy: {best_val_acc:.4f}")

    # Save Keras model
    MODEL_DIR = '/content/psl_models'
    os.makedirs(MODEL_DIR, exist_ok=True)
    h5_path = os.path.join(MODEL_DIR, 'signbridge_psl_gru.h5')
    model.save(h5_path)

    # Save labels
    labels_json = {
        'labels': unique_labels,
        'val_accuracy': float(best_val_acc),
        'num_classes': len(unique_labels),
        'feature_size': FEATURE_SIZE,
        'window_size': WINDOW_SIZE,
    }
    labels_path = os.path.join(MODEL_DIR, 'labels.json')
    with open(labels_path, 'w') as f:
        json.dump(labels_json, f, indent=2)

    print(f"Model saved: {h5_path}  ({os.path.getsize(h5_path) / 1024**2:.1f} MB)")
    print(f"Labels saved: {labels_path}")

In [ ]:
# ── Cell 10: Export to TensorFlow.js ────────────────────────────────────
# Converts the .h5 Keras model to TF.js LayersModel format that the browser
# can load via tf.loadLayersModel().

TFJS_OUT = '/content/psl_models/tfjs_model/psl-gru-word-signs'

!tensorflowjs_converter \
    --input_format=keras \
    --output_format=tfjs_layers_model \
    /content/psl_models/signbridge_psl_gru.h5 \
    {TFJS_OUT}

# Copy vocab.json
shutil.copy(
    '/content/psl_models/labels.json',
    os.path.join(TFJS_OUT, 'vocab.json')
)

print(f"✅ TF.js export complete: {TFJS_OUT}")
print()
for f in sorted(os.listdir(TFJS_OUT)):
    size = os.path.getsize(os.path.join(TFJS_OUT, f))
    print(f"  {f:40s} {size:>8,} bytes")

In [ ]:
# ── Cell 11: Verify TF.js model ─────────────────────────────────────────
MODEL_JSON = os.path.join(TFJS_OUT, 'model.json')
VOCAB_JSON = os.path.join(TFJS_OUT, 'vocab.json')

with open(MODEL_JSON) as f:
    model_meta = json.load(f)
with open(VOCAB_JSON) as f:
    vocab_meta = json.load(f)

total_size = sum(
    os.path.getsize(os.path.join(TFJS_OUT, f))
    for f in os.listdir(TFJS_OUT)
)

print(f"✅ TF.js model verified")
print(f"   model.json:  {os.path.getsize(MODEL_JSON):,} bytes")
print(f"   total size:  {total_size / 1024:.1f} KB")
print(f"   format:      {model_meta.get('format', 'unknown')}")
print(f"   words:       {len(vocab_meta.get('labels', []))}")
print(f"   val acc:     {vocab_meta.get('val_accuracy', 'N/A')}")
print()
print("Labels:")
for i, label in enumerate(vocab_meta.get('labels', [])):
    print(f"  {i:2d}: {label}")

In [ ]:
# ── Cell 12: Save to Google Drive ──────────────────────────────────────
DEST = os.path.join(DRIVE_OUTPUT, 'psl-gru-word-signs')

if os.path.exists(DEST):
    shutil.rmtree(DEST)
shutil.copytree(TFJS_OUT, DEST)

# Write a summary
from datetime import datetime
summary_path = os.path.join(DRIVE_OUTPUT, 'psl_training_summary.txt')
with open(summary_path, 'w') as f:
    f.write("SignBridge PSL — GRU Word-Sign Model\n")
    f.write(f"Trained: {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}\n")
    f.write(f"Dataset: Dynamic Word-Level PSL (Kaggle)\n")
    f.write(f"Architecture: GRU(128)→Dropout(0.3)→GRU(64)→Dropout(0.3)→Softmax\n")
    f.write(f"Input: 30 frames × 159 features (both hands + upper-body pose)\n")
    f.write(f"Classes: {vocab_meta.get('labels', [])}\n")
    f.write(f"Validation accuracy: {vocab_meta.get('val_accuracy', 'N/A')}\n")
    f.write(f"Model size: {total_size / 1024:.1f} KB\n")

print(f"✅ Saved to Google Drive: {DEST}")
print()
print("Files:")
for f in sorted(os.listdir(DEST)):
    size = os.path.getsize(os.path.join(DEST, f))
    print(f"  {f:40s} {size:>8,} bytes")

---
## Next Steps

### 1. Download the model from Google Drive
The `psl-gru-word-signs/` folder is at `MyDrive/signbridge-psl-words/psl-gru-word-signs/`.
Download it to your local machine.

### 2. Copy into the SignBridge web app
Place the folder at:
```
SignBridge/web/public/models/psl-gru-word-signs/
├── model.json
├── group1-shard1of1.bin
└── vocab.json
```

### 3. Verify label alignment
The model labels should match your `pslVocab.ts` PSL_WORD_SIGNS array.
If the trained model has MORE words than your vocab (the dataset has 60+ words),
add the extra words to `pslVocab.ts`.

### 4. Deploy
```bash
cd SignBridge/web
npm run build
vercel --prod
```

### 5. Hand-record missing signs
If some PSL word signs from your vocabulary are missing in the model,
use the SignBridge Data Studio to record them, then re-run training
with the combined dataset.

In [ ]:
# ── Cell 13: Print model labels for vocab.ts verification ──────────────
print("Model output label order (index → label):")
print()
for i, label in enumerate(vocab_meta['labels']):
    print(f"  {i:2d}: '{label}'")

print()
print("Compare these with your PSL_WORD_SIGNS in pslVocab.ts.")
print("The model labels list should include ALL words in your vocab + extras.")
print("The web app's GRU classifier only predicts labels in the loaded model.")
print("Extra words the model knows but aren't in vocab just won't show up in the learning gallery.")